In [ ]:
import os
import sys

import pandas as pd
import py7zr
import rarfile

from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.get_sha256_of_directories import (
    SEPARATOR,
    analyze_file,
    analyze_rar_file,
    analyze_sevenzip_file,
    analyze_tar_file,
    analyze_zip_file,
)
from pynxtools_em.examples.oasisb_utils import CSV_HEADER_FOR_HASH_FILE, get_project_id

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)

## Compute hash values of each file of the projects

Using hash values practically resolves issues when different researchers name their files the same.<br>

In [ ]:
config: dict[str, str] = {
    "python_version": f"{sys.version.replace(' ', '_')}",
    "working_directory": f"{os.getcwd()}",
    "pynxtools_em version": f"{get_pynxtools_em_version()}",
    "rarfile version": f"{rarfile.__version__}",
    "sevenzip version": f"{py7zr.__version__}",
    # "blake3-py version": f"{blake3.__version__}",
    # "blake3-py max_threads": f"{blake3.blake3.AUTO}",
    # "directory": f"src_directory,  # sys.argv[1],
}

spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")
project_range: tuple[int, int] = (1, 836)

# project_name_whitelist = sorted([])
# project_name_blacklist = sorted(())

for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.legal == "0" and row.use == "1":
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.project_name) <= project_range[1]:
            project_id = get_project_id(f"{row.project_name}")

            # if os.path.isfile(f"{src_directory}{os.sep}{project_id}.sha256.results.csv") and os.path.isfile(f"{src_directory}{os.sep}{project_id}.sha256.issues.csv"):
            #     print(f"{row.project_name} has already been analyzed")
            #     continue
            # if int(row.project_name) not in project_name_whitelist:
            #     continue
            # if int(row.project_name) in project_name_blacklist:
            #     continue

            print(f"project{SEPARATOR}{project_id}{SEPARATOR}hashing...")
            sub_directory = f"{src_directory}{os.sep}{project_id}"
            results = []
            issues = []
            project_config = config
            prefix = f"{sub_directory}{os.sep}"
            project_config["directory"] = prefix
            for key, value in project_config.items():
                results.append(f"{key}{SEPARATOR}{value}")
                issues.append(f"{key}{SEPARATOR}{value}")
            del project_config, key, value
            results.append(CSV_HEADER_FOR_HASH_FILE)
            for root, dirs, files in os.walk(sub_directory):
                for file in files:
                    fpath = f"{root}/{file}".replace(os.sep * 2, os.sep)
                    # fname = os.path.basename(fpath)
                    suffix = fpath.replace(config["directory"], "")

                    if fpath.lower().endswith((".zip", ".eln")):
                        analyze_zip_file(fpath, results, issues, prefix)
                    elif fpath.lower().endswith(
                        (".tar", ".tar.gz", ".tar.bz2", ".tar.xz")
                    ):
                        analyze_tar_file(fpath, results, issues, prefix)
                    elif fpath.lower().endswith(".rar"):
                        analyze_rar_file(fpath, results, issues, prefix)
                    elif fpath.lower().endswith(".7z"):
                        analyze_sevenzip_file(fpath, results, issues, prefix)
                    else:
                        analyze_file(fpath, results, issues, prefix)
            del root, dirs, files, file, fpath, suffix
            # for name in ["root", "dirs", "files", "file", "fpath", "suffix"]:
            #     globals().pop(name, None)

            with open(
                f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
                "w",
                encoding="utf-8",
                errors="surrogateescape",
            ) as fp:
                fp.write("\n".join(results))
            del results
            with open(
                f"{src_directory}{os.sep}{project_id}.sha256.issues.csv",
                "w",
                encoding="utf-8",
                errors="surrogateescape",
            ) as fp:
                fp.write("\n".join(issues))
            if len(issues) > 6:
                print(issues)
            else:
                print(f"project{SEPARATOR}{project_id}{SEPARATOR}no issues")
            del issues, sub_directory, prefix

print("Batch queue completed")